库

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

from torch.utils.data import ConcatDataset,DataLoader, Subset
from torchvision.datasets import DatasetFolder

from tqdm.auto import tqdm

要适当做数据的增强,并进行数据的loader

In [ ]:
train_tfm = transforms.Compose([
    transforms.Resize((128, 128)),#调整图像大小
    transforms.RandomHorizontalFlip(), #水平翻转
    transforms.RandomRotation(15),#随机旋转
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),#转换为张量
])

test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),#调整图像大小
    transforms.ToTensor(),#转换为张量
])

batch_size = 128

train_set = DatasetFolder(root='./data/train', loader=lambda x: Image.open(x), extensions='jpg', transform=train_tfm)
valid_set = DatasetFolder(root='./data/test', loader=lambda x: Image.open(x), extensions='jpg', transform=test_tfm)
unlabeled_set = DatasetFolder(root='./data/unlabeled', loader=lambda x: Image.open(x), extensions='jpg', transform=train_tfm)
test_set = DatasetFolder(root="food_11/testing", loader=lambda x: Image.open(x), extensions='jpg', transform=test_tfm)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8,pin_memory=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, num_workers=8,pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

CNN model

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),#卷积层
            nn.BatchNorm2d(64),#批归一化，batch normalization
            nn.ReLU(),#激活函数,conv->BN->ReLU->pool
            nn.MaxPool2d(2,2,0),#池化层
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2,0),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(4,4,0),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(256*8*8, 256),#全连接层,256为通道数
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 11),
        )

    def forward(self, x):
        x = self.cnn_layers(x)
        x = x.flatten(start_dim=1)#展平
        x = self.fc_layers(x)
        return x

半自助监督学习，即贴加标签

In [ ]:
def get_pseudo_label(dataset, model, threshold=0.65):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    
    softmax = nn.Softmax(dim=-1)
    
    pesudo_images = []
    pesudo_labels = []
    
    for batch in tqdm(data_loader):
        images, _ = batch
        images = images.to(device)
        
        with torch.no_grad():
            logits = model(images)
        
        probs = softmax(logits)
        max_probs, pseudo_labels = torch.max(probs, dim=-1)#获取最大概率及其对应的标签,作为伪标签
        mask = max_probs >= threshold
        pesudo_images.append(images[mask].cpu())
        pesudo_labels.append(labels[mask].cpu())
        
    pesudo_images = torch.cat(pesudo_images, dim=0)
    pesudo_labels = torch.cat(pesudo_labels, dim=0)
    
    pseudo_dataset = torch.utils.data.TensorDataset(pesudo_images, pesudo_labels)
        
    model.train()
    return pseudo_dataset

TRAINING

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = Classifier().to(device)
model.device = device

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003,weight_decay=1e-5)

n_epochs = 80

do_semi_supervised = False

for epoch in range(n_epochs):
    if do_semi_supervised :
        pseudo_set = get_pseudo_label(unlabeled_set, model)
        concat_dataset = ConcatDataset([train_set, pseudo_set])
        train_loader = DataLoader(concat_dataset, batch_size=batch_size, shuffle=True, num_workers=8,pin_memory=True)
    
    model.train()
    
    train_loss = []
    train_accs = []
    
    for batch in tqdm(train_loader):
        images, labels = batch
        logits =model(images.to(device))#前向传播
        loss=criterion(logits, labels.to(device))#计算损失
        
        optimizer.zero_grad()#清空梯度
        
        loss.backward()#反向传播，计算梯度
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)#梯度裁剪，防止梯度爆炸
        optimizer.step()#更新参数，用Adam优化器
        
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()
        
        train_loss.append(loss.item())
        train_accs.append(acc.item())

    # The average loss and accuracy of the training set is the average of the recorded values.
    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)

    # Print the information.
    print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

    # ---------- Validation ----------
    # Make sure the model is in eval mode so that some modules like dropout are disabled and work normally.
    model.eval()

    # These are used to record information in validation.
    valid_loss = []
    valid_accs = []

    # Iterate the validation set by batches.
    for batch in tqdm(valid_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch

        # We don't need gradient in validation.
        # Using torch.no_grad() accelerates the forward process.
        with torch.no_grad():
          logits = model(imgs.to(device))

        # We can still compute the loss (but not the gradient).
        loss = criterion(logits, labels.to(device))

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # Record the loss and accuracy.
        valid_loss.append(loss.item())
        valid_accs.append(acc)

    # The average loss and accuracy for entire validation set is the average of the recorded values.
    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)

    # Print the information.
    print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")


In [ ]:
# Make sure the model is in eval mode.
# Some modules like Dropout or BatchNorm affect if the model is in training mode.
model.eval()

# Initialize a list to store the predictions.
predictions = []

# Iterate the testing set by batches.
for batch in tqdm(test_loader):
    # A batch consists of image data and corresponding labels.
    # But here the variable "labels" is useless since we do not have the ground-truth.
    # If printing out the labels, you will find that it is always 0.
    # This is because the wrapper (DatasetFolder) returns images and labels for each batch,
    # so we have to create fake labels to make it work normally.
    imgs, labels = batch

    # We don't need gradient in testing, and we don't even have labels to compute loss.
    # Using torch.no_grad() accelerates the forward process.
    with torch.no_grad():
        logits = model(imgs.to(device))

    # Take the class with greatest logit as prediction and record it.
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
     

# Save predictions into the file.
with open("predict.csv", "w") as f:

    # The first row must be "Id, Category"
    f.write("Id,Category\n")

    # For the rest of the rows, each image id corresponds to a predicted class.
    for i, pred in  enumerate(predictions):
         f.write(f"{i},{pred}\n")